# 01 - Data acquisition and EDA
Heart Disease UCI (Cleveland, 303 patients). Downloads the raw file, cleans it and produces the EDA figures used in the report.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 140)

In [ ]:
from heartrisk import data, eda, config
raw_path = data.download()
raw = data.load_raw()
print(raw.shape)
raw.head()

## Data quality checks
The raw file marks missing values with `?`. Only `ca` (4 rows) and `thal` (2 rows) have gaps. `num` ranges 0-4, so it is converted to a binary target (0 = no disease, 1-4 = disease).

In [ ]:
print(raw.isna().sum()[raw.isna().sum() > 0])
print('\nraw target (num) counts:'); print(raw['num'].value_counts().sort_index())
raw.describe().T.round(2)

In [ ]:
df = data.make_clean_dataset()
print('clean shape', df.shape, '| duplicates removed:', len(raw) - len(df))
df[config.TARGET].value_counts()

Missing values are left as NaN in the cleaned file on purpose. They are imputed inside the sklearn pipeline (median / most frequent), so the imputer only learns from training folds and the same step runs in the API.

In [ ]:
info = eda.run(df)
info

## Class balance
164 vs 139 (54% / 46%) - mildly imbalanced, so plain accuracy is acceptable but recall and ROC-AUC are tracked as well.

In [ ]:
display(Image('reports/figures/01_class_balance.png'))

## Distributions by outcome
Patients with disease reach a lower max heart rate (`thalach`) and have higher ST depression (`oldpeak`). Cholesterol separates the classes very weakly.

In [ ]:
display(Image('reports/figures/02_histograms.png'))
display(Image('reports/figures/05_boxplots_by_outcome.png'))

## Correlations
`thal`, `ca`, `exang`, `oldpeak`, `thalach` and `cp` have the strongest link with the target. No pair of inputs is so correlated that it needs to be dropped.

In [ ]:
display(Image('reports/figures/03_correlation_heatmap.png'))

## Categorical features
Asymptomatic chest pain (`cp=4`) has a ~73% disease rate - counter-intuitive but well known for this dataset. `restecg=1` has only 4 patients, so one-hot encoding uses `handle_unknown='ignore'` to stay safe.

In [ ]:
display(Image('reports/figures/04_categorical_disease_rate.png'))